# Ensemble Inference — 3-Fold OOF 앙상블

**전제**: `colab_train_oof.ipynb` 완료 후 `lora_model_fold_0~2` 업로드 필요

다수결 앙상블로 최종 submission.csv 생성

## 1. Install Dependencies
**실행 후 Runtime > Restart Session 필수**

In [ ]:
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps unsloth_zoo
!pip install --upgrade "transformers==5.5.0" "trl==0.24.0" "datasets<4.4.0"
!pip install cut_cross_entropy hf_transfer msgspec tyro peft accelerate bitsandbytes xformers
!pip install flash-attn --no-build-isolation
!pip install pandas tqdm scikit-learn sentence-transformers

## 2. 파일 업로드 확인
업로드 필요:
- `my_code_0514from0508/` 폴더
- `data/` 폴더
- `lora_model_fold_0/`, `lora_model_fold_1/`, `lora_model_fold_2/` (또는 압축 해제)

In [ ]:
import os
os.chdir('/content')

SRC_DIR = '/content/my_code_0514from0508'

# lora_oof_folds.zip 업로드했으면 압축 해제
if os.path.exists('/content/lora_oof_folds.zip'):
    !unzip -o lora_oof_folds.zip

for fold in range(3):
    path = f'/content/lora_model_fold_{fold}'
    exists = os.path.exists(path)
    print(f'lora_model_fold_{fold}: {"OK" if exists else "MISSING"}')

## 3. GPU 확인

In [ ]:
import torch
print(f'GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB')

## 4. 앙상블 추론 실행

In [ ]:
import subprocess, sys
result = subprocess.run(
    [sys.executable, f'{SRC_DIR}/inference.py', '--ensemble'],
    cwd='/content',
)
print('Return code:', result.returncode)

## 5. 제출 파일 다운로드

In [ ]:
from google.colab import files
!find /content -name 'submission.csv' 2>/dev/null

for path in ['/content/submission.csv', '/content/artifacts/submission.csv']:
    if os.path.exists(path):
        import pandas as pd
        df = pd.read_csv(path)
        print(f'{path}: {len(df)} rows')
        print(f'op dist: {dict(df["op"].value_counts())}')
        empty = (df['target_id'].astype(str).str.strip() == '').sum()
        print(f'empty target_id: {empty}')
        files.download(path)
        break